# 03 — Regression: hyperparameter tuning

Tunes eight regression models with Optuna, each maximising **Quadratic Weighted
Kappa** over 5-fold stratified cross-validation.

The models predict `sii` as a continuous number, so every fold has to cut that
number into the four ordinal classes before scoring. Those cut points are
themselves optimised — on the training fold's predictions only — and then
applied to the validation fold.

Each fold re-fits its own scaler, imputer and PCA. That is the part that keeps
the scores honest: fitting them once on the whole training set would let
validation rows influence the model.

Every study writes its full trial history to `optuna trials/<model>_reg_trials.csv`,
which notebook 04 reads back.

Equivalent to: `python main.py --stage tune-reg`

**This is the slow stage** — the full sweep is several hours. Use
`tune_model(..., n_trials=5)` to try it out quickly.

## Setup and data loading

In [1]:
# Make the project's src/ package importable from inside notebooks/.
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

import warnings

from src.config import IMPUTER_PARAMS_REG, PATHS, STUDY_TRIALS
from src.data_loader import DROP_FOR_TUNING, load_processed_split
from src.tuning import tune_model

warnings.filterwarnings("ignore")
os.makedirs(PATHS["results_dir"], exist_ok=True)

# Load the dataset built by 02a. Features stay RAW here — no imputation or
# scaling yet, because those happen inside each CV fold.
# Note that `is_outlier` is kept as an ordinary feature at tuning time; the
# final evaluation in notebook 04 drops it instead.
X_train_raw, y_train_cv, _, _ = load_processed_split(
    PATHS["reg_processed"], DROP_FOR_TUNING
)

print(f"X_train_raw shape: {X_train_raw.shape} "
      f"(NaNs present: {X_train_raw.isnull().any().any()})")
print(f"Active imputer configuration: {IMPUTER_PARAMS_REG}")

/Users/alessandro/miniforge3/envs/m4_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


X_train_raw shape: (6768, 46) (NaNs present: True)
Active imputer configuration: {'imputer_choice': 'mice', 'mice_max_iter': 10, 'mice_initial_strategy': 'median', 'mice_et_estimators': 20, 'mice_et_max_depth': 4}


## LightGBM — 50 trials

Gradient-boosted trees. Given the largest budget (50 trials) because it was the strongest single model.

In [2]:
study_lgbm = tune_model("lgbm", "reg", X_train_raw, y_train_cv,
                        IMPUTER_PARAMS_REG, PATHS["results_dir"])

Starting LightGBM optimization (maximizing QWK)...


Best trial: 46. Best value: 0.288628: 100%|██████████| 50/50 [1:56:00<00:00, 139.21s/it]

LightGBM: Best trial CV QWK: 0.2886


## XGBoost — 20 trials

A second boosting implementation with different regularisation defaults.

In [3]:
study_xgb = tune_model("xgb", "reg", X_train_raw, y_train_cv,
                        IMPUTER_PARAMS_REG, PATHS["results_dir"])

Starting XGBoost optimization (maximizing QWK)...


Best trial: 18. Best value: 0.280135: 100%|██████████| 20/20 [43:31<00:00, 130.55s/it]

XGBoost: Best trial CV QWK: 0.2801


## CatBoost — 20 trials

Boosting with ordered target statistics, which tends to overfit small data less.

In [4]:
study_cb = tune_model("cb", "reg", X_train_raw, y_train_cv,
                        IMPUTER_PARAMS_REG, PATHS["results_dir"])

Starting CatBoost optimization (maximizing QWK)...


Best trial: 18. Best value: 0.237984: 100%|██████████| 20/20 [47:15<00:00, 141.80s/it]

CatBoost: Best trial CV QWK: 0.2380


## Random Forest — 20 trials

Bagged trees — lower variance than boosting, and a useful sanity baseline.

In [5]:
study_rf = tune_model("rf", "reg", X_train_raw, y_train_cv,
                        IMPUTER_PARAMS_REG, PATHS["results_dir"])

Starting RandomForest optimization (maximizing QWK)...


Best trial: 1. Best value: 0.308859: 100%|██████████| 20/20 [51:11<00:00, 153.55s/it]

RandomForest: Best trial CV QWK: 0.3089


## Ridge — 20 trials

Linear baseline. Tuned jointly with a PCA step, since a linear model on 45 correlated features is unstable.

In [6]:
study_ridge = tune_model("ridge", "reg", X_train_raw, y_train_cv,
                        IMPUTER_PARAMS_REG, PATHS["results_dir"])

Starting Ridge optimization (maximizing QWK)...


Best trial: 0. Best value: 0.267423: 100%|██████████| 20/20 [44:49<00:00, 134.49s/it]

Ridge: Best trial CV QWK: 0.2674


## SVR — 20 trials

Kernel regression, also tuned jointly with PCA. The kernel choice is itself a search parameter.

In [7]:
study_svr = tune_model("svr", "reg", X_train_raw, y_train_cv,
                        IMPUTER_PARAMS_REG, PATHS["results_dir"])

Starting SVR optimization (maximizing QWK)...


Best trial: 15. Best value: 0.264274: 100%|██████████| 20/20 [1:04:55<00:00, 194.77s/it]

SVR: Best trial CV QWK: 0.2643


## PyTorch MLP — 20 trials

The neural network. PCA is capped at 45 components here — with this little data a wider input overfits fast.

In [8]:
study_nn = tune_model("nn", "reg", X_train_raw, y_train_cv,
                        IMPUTER_PARAMS_REG, PATHS["results_dir"])

Starting TorchMLP optimization (maximizing QWK)...


Best trial: 3. Best value: 0.305964: 100%|██████████| 20/20 [1:14:06<00:00, 222.30s/it]

TorchMLP: Best trial CV QWK: 0.3060


## Lasso — 20 trials

L1 linear baseline, tuned with PCA. Included to see whether sparsity helps.

In [9]:
study_lasso = tune_model("lasso", "reg", X_train_raw, y_train_cv,
                        IMPUTER_PARAMS_REG, PATHS["results_dir"])

Starting Lasso optimization (maximizing QWK)...


Best trial: 17. Best value: 0.267302: 100%|██████████| 20/20 [48:09<00:00, 144.50s/it]

Lasso: Best trial CV QWK: 0.2673
